Taking a look at unmapped reads

In [ ]:
from Bio import SeqIO

# import the unmapped reads (end2end and local)
end2end_path = (
    "/home/lawrence/Documents/School/thesis/"
    "PFAS_project_preprocessing_pipeline/Results_from_pipeline/"
    "bowtie/ref/SRR29882081_end2end_unmapped.fastq"
)

local_path = (
    "/home/lawrence/Documents/School/thesis/"
    "PFAS_project_preprocessing_pipeline/Results_from_pipeline/"
    "bowtie/ref/SRR29882081_local_unmapped.fastq"
)

end2end_reads = list(SeqIO.parse(end2end_path, "fastq"))
local_reads = list(SeqIO.parse(local_path, "fastq"))

print(f"End-to-end reads: {len(end2end_reads)}")
print(f"Local reads: {len(local_reads)}")

# Example: first sequence
print(end2end_reads[0].id)
print(end2end_reads[0].seq)

End-to-end reads: 600806
Local reads: 447151
SRR29882081.6
GTTCATCCACCAACCAGATCCATCCTGTCAGCACAGTTGCCTTCAAACTCC


In [3]:
end2end_reads

[SeqRecord(seq=Seq('GTTCATCCACCAACCAGATCCATCCTGTCAGCACAGTTGCCTTCAAACTCC'), id='SRR29882081.6', name='SRR29882081.6', description='SRR29882081.6 NB501400:173:HF2GWBGXL:1:11101:21829:1140 length=51', dbxrefs=[]),
 SeqRecord(seq=Seq('CCACTGGGCTGACGTCATTGAGGAGGTCCTAAATCATCCTCCTGTGGTCGT'), id='SRR29882081.7', name='SRR29882081.7', description='SRR29882081.7 NB501400:173:HF2GWBGXL:1:11101:16380:1147 length=51', dbxrefs=[]),
 SeqRecord(seq=Seq('AAGGATGGCAGGCACCCACGTACAGGGTCCTGCCATCCTGTGGTCGTAGCA'), id='SRR29882081.11', name='SRR29882081.11', description='SRR29882081.11 NB501400:173:HF2GWBGXL:1:11101:18305:1228 length=51', dbxrefs=[]),
 SeqRecord(seq=Seq('TCAGGATGCTCTGGGAGCTAAGCTCCCAGGCAAACTAAACGAGACCTCTGG'), id='SRR29882081.20', name='SRR29882081.20', description='SRR29882081.20 NB501400:173:HF2GWBGXL:1:11101:22756:1328 length=51', dbxrefs=[]),
 SeqRecord(seq=Seq('TAGTGCTCCCTGTGGTCGTAGCATCAGCTAGTAGGACAGACACTCATGTAA'), id='SRR29882081.21', name='SRR29882081.21', description='SRR29882081.21 NB5

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from Bio import SeqIO

UCSC_API_KEY = os.environ["UCSC_API_KEY"]

fastq_files = [
    Path("/home/lawrence/Documents/School/thesis/"
         "PFAS_project_preprocessing_pipeline/Results_from_pipeline/"
         "bowtie/ref/SRR29882081_end2end_unmapped.fastq"),
    Path("/home/lawrence/Documents/School/thesis/"
         "PFAS_project_preprocessing_pipeline/Results_from_pipeline/"
         "bowtie/ref/SRR29882081_local_unmapped.fastq"),
]

results = []
query_count = 0

for fastq_path in fastq_files:
    for record in SeqIO.parse(fastq_path, "fastq"):
        if query_count:
            time.sleep(15)  # UCSC programmatic BLAT limit

        response = requests.get(
            "https://genome.ucsc.edu/cgi-bin/hgBlat",
            params={
                "userSeq": str(record.seq),
                "type": "DNA",
                "db": "hg38",
                "output": "json",
                "apiKey": UCSC_API_KEY,
            },
            timeout=120,
        )
        response.raise_for_status()

        blat_result = response.json()
        hits = blat_result.get("blat", [])

        results.append({
            "source_file": fastq_path.name,
            "read_id": record.id,
            "sequence": str(record.seq),
            "found": len(hits) > 0,
            "number_of_hits": len(hits),
            "blat_results": hits,
        })

        query_count += 1
        print(
            f"{query_count}: {record.id} | "
            f"{'FOUND' if hits else 'NOT FOUND'} | "
            f"{len(hits)} hit(s)"
        )

results_df = pd.DataFrame(results)

Count found versus not found:

In [ ]:
results_df["found"].value_counts()

Display the summary:

In [ ]:
summary_df = pd.DataFrame({
    "total_sequences": [len(results_df)],
    "sequences_with_hits": [results_df["found"].sum()],
    "sequences_without_hits": [(~results_df["found"]).sum()],
})

display(summary_df)

View sequences with no BLAT results:

In [ ]:
not_found_df = results_df[~results_df["found"]]

display(not_found_df[[
    "source_file",
    "read_id",
    "sequence",
    "number_of_hits",
]])

View the first BLAT alignment result:

In [ ]:
display(results_df[results_df["found"]].iloc[0]["blat_results"])